# Notebook 8: Exportar datos para Power BI
## Transformar datos a Star Schema y exportar CSVs

**Objetivo:** Generar 5 CSVs en `data/power_bi_export/` con un modelo estrella (star schema) listo para importar en Power BI Desktop.

**Tú escribes todo el código.** Aquí solo tienes las instrucciones de lo que debe hacer cada celda.

## Celda 1: Importar librerías

Importa pandas, numpy, y sys para cargar los módulos de src/.

In [12]:
import pandas as pd
import numpy as np
import os
import sys #sirve para obtener argumentos de la linea de comandos
sys.path.append('..') #sirve para importar modulos de carpetas superiores




## Celda 2: Cargar datos limpios

Carga los 5 CSVs desde `data/processed/` usando `pd.read_csv()`.

In [13]:
editions = pd.read_csv('../data/processed/editions_clean.csv')
matches = pd.read_csv('../data/processed/matches_clean.csv')
top_scorers = pd.read_csv('../data/processed/top_scorers_clean.csv')
teams_2026 = pd.read_csv('../data/processed/teams_2026_clean.csv')
fixtures = pd.read_csv('../data/processed/fixtures_2026_clean.csv')


## Celda 3: Crear dim_teams

Extraer todos los equipos únicos de `fact_editions` (champion, runner_up, third_place, fourth_place) y de `fact_matches` (team1, team2).

Asignar confederación a cada equipo (mapeo manual).
Contar títulos, subcampeonatos y apariciones en top 4.
Exportar a `data/power_bi_export/dim_teams.csv`

In [14]:
# Extraer equipos únicos desde matches (todos los partidos)
top4 = pd.concat([
    editions['champion'],
    editions['runner_up'],
    editions['third_place'],
    editions['fourth_place']
]).unique()

#Unir ambas listas y crear DataFrame base

all_teams = pd.concat([
    matches['team1'], # Obtener todos los equipos únicos
    matches['team2'],]).unique() # Obtener todos los equipos únicos


#Calcular títulos por equipo
all_teams = sorted(set(list(top4) + list(all_teams) + list(teams_2026['team']))) # Obtener todos los equipos únicos
teams_df = pd.DataFrame({'team_name': all_teams}) # Crear DataFrame con los nombres de los equipos
teams_df['team_id'] =  range(1, len(teams_df) + 1) # Asignar un ID único a cada equipo


#Calcular subcampeonatos por equipo
runnerup_counts = editions['runner_up'].value_counts().reset_index() #reset_index() para convertir el índice en una columna y value counts() para contar la cantidad de veces que cada equipo ha sido subcampeón
runnerup_counts.columns = ['team_name', 'runner_up'] # Renombrar columnas

teams_df = teams_df.merge(runnerup_counts, on='team_name', how='left') # Unir con el DataFrame de equipos
teams_df['runner_up'] = teams_df['runner_up'].fillna(0).astype(int)


#Calcular tercer lugar por equipo
third_place_counts = editions['third_place'].value_counts().reset_index() #reset_index()
third_place_counts.columns = ['team_name', 'third_place'] # Renombrar columnas

teams_df = teams_df.merge(third_place_counts, on='team_name', how='left') # Unir con el DataFrame de equipos
teams_df['third_place'] = teams_df['third_place'].fillna(0).astype(int)

#Calcular cuarto lugar por equipo
fourth_place_counts = editions['fourth_place'].value_counts().reset_index() #reset_index()
fourth_place_counts.columns = ['team_name', 'fourth_place'] # Renombrar columnas

teams_df = teams_df.merge(fourth_place_counts, on='team_name', how='left') # Unir con el DataFrame de equipos
teams_df['fourth_place'] = teams_df['fourth_place'].fillna(0).astype(int)

#agregar datos de 2026
teams_2026_slim = teams_2026[['team', 'confederation', 'fifa_rank']].rename(columns={'team': 'team_name', 'fifa_rank': 'fifa_rank_2026'})
teams_df = teams_df.merge(teams_2026_slim, on='team_name', how='left') # Unir con el DataFrame de equipos


os.makedirs('../data/power_bi_export', exist_ok=True) # Crear carpeta si no existe
teams_df.to_csv('../data/power_bi_export/dim_teams.csv', index=False) # Guardar DataFrame en CSV

print("Exported dim_teams.csv to ../data/power_bi_export/")

Exported dim_teams.csv to ../data/power_bi_export/


## Celda 4: Crear dim_confederations

Crear DataFrame con las 6 confederaciones: UEFA, CONMEBOL, CAF, AFC, CONCACAF, OFC.
Asignar un color hexadecimal a cada una para usar en Power BI.
Exportar a `data/power_bi_export/dim_confederations.csv`

In [18]:
confederations = pd.DataFrame({
    'confederation_id': ['UEFA', 'CONMEBOL', 'CONCACAF', 'CAF', 'AFC', 'OFC'],
    'confederation_name': [
        'Union of European Football Associations',
        'Confederación Sudamericana de Fútbol',
        'Confederación Norte, Centroamericana y del Caribe de Fútbol',
        'Confédération Africaine de Football', 'Asian Football Confederation',
        'Oceania Football Confederation'
    ],
    'color_hex': ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6', '#1abc9c'] # Asignar un color hexadecimal a cada confederación
})

os.makedirs('../data/power_bi_export', exist_ok=True) # Crear carpeta si no existe
confederations.to_csv('../data/power_bi_export/dim_confederations.csv', index=False) # Guardar DataFrame en CSV

## Celda 5: Crear dim_calendar

Generar una tabla calendario con todos los años desde 1930 hasta 2026.
Columnas: date_id (YYYYMMDD), year, decade, month, day.
Exportar a `data/power_bi_export/dim_calendar.csv`

In [19]:
years = list(range(1930, 2027)) # Generar lista de años desde 1930 hasta 2026 con un paso de 4 años
cal = pd.DataFrame({'year': years}) # Crear DataFrame con los años
cal['decade'] = (cal['year'] // 10) * 10 # Calcular la década a partir del año
cal['date_id'] = cal['year'] * 10000 + 101 # Crear un ID de fecha en formato YYYYMMDD, donde MM y DD son siempre 01
cal['month'] = 1 # Asignar el mes como 1 (enero)
cal['day'] = 1 # Asignar el día como 1

os.makedirs('../data/power_bi_export', exist_ok=True) # Crear carpeta si no existe
cal.to_csv('../data/power_bi_export/dim_calendar.csv', index=False) # Guardar DataFrame en CSV

## Celda 6: Verificar archivos exportados

Listar los archivos en `data/power_bi_export/` y mostrar las primeras filas de cada uno para verificar que todo está correcto.

---

## Siguiente paso

Una vez exportados los CSVs, abre **Power BI Desktop** y sigue la guía en `power_bi/POWER_BI_GUIDE.md` para:
1. Importar los CSVs
2. Crear las relaciones
3. Escribir las medidas DAX
4. Construir las 6 páginas del dashboard